# Notebook 4: Creating Historical Data (Improved)

This notebook builds a 15-minute region-wise historical dataset and adds business-intelligence features for driver decision support.

## Core outputs
- demand time series (`total_pickups_raw`, smoothed demand, rolling stats)
- surge detection (`surge_flag`, `surge_level`)
- risk/stability (`risk_score`, `risk_band`)
- revenue estimation (`expected_revenue`, `fare_per_km`)
- driver relocation recommendation (`recommended_next_zone`, `relocation_score`)
- best-time recommendation table per region


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

data_interim = project_root / "data" / "interim"
data_processed = project_root / "data" / "processed"

for folder in [data_interim, data_processed]:
    folder.mkdir(parents=True, exist_ok=True)

candidate_inputs = [
    data_interim / "region_labeled_data.csv",
    data_interim / "time_series.csv",
    data_interim / "processing_data.csv",
]

input_path = next((p for p in candidate_inputs if p.exists()), None)
if input_path is None:
    raise FileNotFoundError(
        "Could not find historical input. Expected one of: " + ", ".join(str(p) for p in candidate_inputs)
    )

neighbors_path = data_interim / "region_neighbors.csv"

print(f"Using input data: {input_path}")
print(f"Using neighbors file: {neighbors_path} (exists={neighbors_path.exists()})")


In [ ]:
# Load dataset
df = pd.read_csv(input_path, low_memory=False)

# Normalize region column name
if "region" in df.columns:
    df["region"] = pd.to_numeric(df["region"], errors="coerce")
elif "region_id" in df.columns:
    df["region"] = pd.to_numeric(df["region_id"], errors="coerce")
else:
    raise ValueError("Missing region column. Expected `region` or `region_id`.")

# Parse datetime
if "tpep_pickup_datetime" not in df.columns:
    raise ValueError("Missing `tpep_pickup_datetime` column in input data.")

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")

df = df.dropna(subset=["tpep_pickup_datetime", "region"]).copy()
df["region"] = df["region"].astype(int)
df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)

# Revenue helper if not already present
if "fare_efficiency" not in df.columns and {"total_amount", "trip_distance"}.issubset(df.columns):
    df["fare_efficiency"] = df["total_amount"] / df["trip_distance"].clip(lower=1e-3)

df[["tpep_pickup_datetime", "region"]].head()


In [ ]:
# Build 15-minute slots
df["pickup_slot"] = df["tpep_pickup_datetime"].dt.floor("15min")

# Base demand count per region/time-slot
grouped_counts = (
    df.groupby(["region", "pickup_slot"], as_index=False)
    .size()
    .rename(columns={"size": "total_pickups_raw"})
)

# Optional aggregates from available columns
metric_agg = {}
if "total_amount" in df.columns:
    metric_agg["total_amount"] = "sum"
if "tip_amount" in df.columns:
    metric_agg["tip_amount"] = "sum"
if "trip_distance" in df.columns:
    metric_agg["trip_distance"] = "sum"
if "trip_duration_min" in df.columns:
    metric_agg["trip_duration_min"] = "mean"
if "passenger_count" in df.columns:
    metric_agg["passenger_count"] = "mean"
if "fare_efficiency" in df.columns:
    metric_agg["fare_efficiency"] = "mean"

if metric_agg:
    grouped_metrics = df.groupby(["region", "pickup_slot"], as_index=False).agg(metric_agg)
    rename_map = {
        "total_amount": "total_revenue",
        "tip_amount": "total_tip",
        "trip_distance": "total_trip_distance",
        "trip_duration_min": "avg_trip_duration_min",
        "passenger_count": "avg_passenger_count",
        "fare_efficiency": "avg_fare_efficiency",
    }
    grouped_metrics = grouped_metrics.rename(columns=rename_map)
    historical = grouped_counts.merge(grouped_metrics, on=["region", "pickup_slot"], how="left")
else:
    historical = grouped_counts.copy()

# Build full region-time grid so missing intervals are explicit
all_regions = np.sort(historical["region"].unique())
all_slots = pd.date_range(historical["pickup_slot"].min(), historical["pickup_slot"].max(), freq="15min")

full_index = pd.MultiIndex.from_product([all_regions, all_slots], names=["region", "pickup_slot"])
historical = historical.set_index(["region", "pickup_slot"]).reindex(full_index).reset_index()

# Fill count/sum columns with 0
zero_fill_cols = [
    "total_pickups_raw",
    "total_revenue",
    "total_tip",
    "total_trip_distance",
]
for col in zero_fill_cols:
    if col in historical.columns:
        historical[col] = historical[col].fillna(0)

# Fill mean-style columns by region ffill/bfill fallback
mean_like_cols = [
    "avg_trip_duration_min",
    "avg_passenger_count",
    "avg_fare_efficiency",
]
for col in mean_like_cols:
    if col in historical.columns:
        historical[col] = historical.groupby("region")[col].transform(lambda s: s.ffill().bfill())
        historical[col] = historical[col].fillna(historical[col].median())

historical = historical.sort_values(["region", "pickup_slot"]).reset_index(drop=True)
historical.head()


In [ ]:
# Keep raw pickups and create model-safe demand (for MAPE stability)
epsilon_val = 10
historical["total_pickups_model"] = historical["total_pickups_raw"].replace(0, epsilon_val)

# Candidate grids for smoothing tuning
ma_windows = [3, 4, 6, 8, 12, 16]
ewma_alphas = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Validation split per region: last 20% (with minimum tail)
tuning_df = historical[["region", "pickup_slot", "total_pickups_model"]].copy()
tuning_df = tuning_df.sort_values(["region", "pickup_slot"]).reset_index(drop=True)
tuning_df["row_num"] = tuning_df.groupby("region").cumcount()
tuning_df["region_size"] = tuning_df.groupby("region")["total_pickups_model"].transform("size")

min_val_points = 24  # 24 points = 6 hours at 15-minute granularity
val_start_idx = np.maximum(
    (tuning_df["region_size"] * 0.8).astype(int),
    tuning_df["region_size"] - min_val_points,
)
tuning_df["is_validation"] = tuning_df["row_num"] >= val_start_idx


def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    return np.mean(2.0 * np.abs(y_true - y_pred) / np.where(denom == 0, 1.0, denom))


ma_results = []
for window in ma_windows:
    pred = tuning_df.groupby("region")["total_pickups_model"].transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
    )

    val_mask = tuning_df["is_validation"] & pred.notna()
    y_true = tuning_df.loc[val_mask, "total_pickups_model"].values
    y_pred = pred.loc[val_mask].values

    ma_results.append(
        {
            "method": "moving_average",
            "param": window,
            "mape": float(mean_absolute_percentage_error(y_true, y_pred)),
            "mae": float(mean_absolute_error(y_true, y_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "smape": float(smape(y_true, y_pred)),
            "eval_points": int(val_mask.sum()),
        }
    )


ewm_results = []
for alpha in ewma_alphas:
    pred = tuning_df.groupby("region")["total_pickups_model"].transform(
        lambda s: s.shift(1).ewm(alpha=alpha, adjust=False).mean()
    )

    val_mask = tuning_df["is_validation"] & pred.notna()
    y_true = tuning_df.loc[val_mask, "total_pickups_model"].values
    y_pred = pred.loc[val_mask].values

    ewm_results.append(
        {
            "method": "ewma",
            "param": alpha,
            "mape": float(mean_absolute_percentage_error(y_true, y_pred)),
            "mae": float(mean_absolute_error(y_true, y_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "smape": float(smape(y_true, y_pred)),
            "eval_points": int(val_mask.sum()),
        }
    )

ma_metrics = pd.DataFrame(ma_results).sort_values(["mape", "mae", "rmse"]).reset_index(drop=True)
ewma_metrics = pd.DataFrame(ewm_results).sort_values(["mape", "mae", "rmse"]).reset_index(drop=True)

best_ma_row = ma_metrics.iloc[0]
best_ewma_row = ewma_metrics.iloc[0]

best_ma_window = int(best_ma_row["param"])
best_ewma_alpha = float(best_ewma_row["param"])

if best_ewma_row["mape"] <= best_ma_row["mape"]:
    selected_smoothing_method = "ewma"
else:
    selected_smoothing_method = "moving_average"

# Build tuned smoothed series
historical["avg_pickups_ma_tuned"] = historical.groupby("region")["total_pickups_model"].transform(
    lambda s: s.rolling(window=best_ma_window, min_periods=1).mean()
)
historical["avg_pickups_ewm_tuned"] = historical.groupby("region")["total_pickups_model"].transform(
    lambda s: s.ewm(alpha=best_ewma_alpha, adjust=False).mean()
)

# One-step-ahead proxy (no leakage)
historical["predicted_demand_proxy_ma"] = historical.groupby("region")["avg_pickups_ma_tuned"].shift(1)
historical["predicted_demand_proxy_ewm"] = historical.groupby("region")["avg_pickups_ewm_tuned"].shift(1)

historical["predicted_demand_proxy_ma"] = historical["predicted_demand_proxy_ma"].fillna(historical["avg_pickups_ma_tuned"])
historical["predicted_demand_proxy_ewm"] = historical["predicted_demand_proxy_ewm"].fillna(historical["avg_pickups_ewm_tuned"])

if selected_smoothing_method == "ewma":
    historical["predicted_demand_proxy"] = historical["predicted_demand_proxy_ewm"]
else:
    historical["predicted_demand_proxy"] = historical["predicted_demand_proxy_ma"]

historical["predicted_demand"] = historical["predicted_demand_proxy"].clip(lower=0)

# Keep compatibility columns
historical["avg_pickups_ewm"] = historical["avg_pickups_ewm_tuned"]
historical["avg_pickups"] = historical["predicted_demand_proxy"]
historical["smoothing_method"] = selected_smoothing_method
historical["selected_ma_window"] = best_ma_window
historical["selected_ewma_alpha"] = best_ewma_alpha

# Time flags for downstream modeling and dashboard filters
historical["pickup_day_of_week"] = historical["pickup_slot"].dt.dayofweek
historical["pickup_hour"] = historical["pickup_slot"].dt.hour
historical["is_weekend"] = historical["pickup_day_of_week"] >= 5
historical["rush_hour"] = historical["pickup_hour"].isin([7, 8, 9, 17, 18, 19])
historical["is_night"] = (historical["pickup_hour"] >= 23) | (historical["pickup_hour"] <= 4)

smoothing_metrics = pd.concat([ma_metrics, ewma_metrics], ignore_index=True)

print("Best MA window:", best_ma_window, "| MAPE:", round(float(best_ma_row["mape"]), 4))
print("Best EWMA alpha:", best_ewma_alpha, "| MAPE:", round(float(best_ewma_row["mape"]), 4))
print("Selected smoothing method:", selected_smoothing_method)


In [ ]:
# Surge detection + risk/stability score
rolling_window = 96  # 96 * 15min = 24 hours
surge_k = 1.5

historical["rolling_mean"] = (
    historical.groupby("region")["total_pickups_model"]
    .transform(lambda s: s.rolling(window=rolling_window, min_periods=16).mean())
)
historical["rolling_std"] = (
    historical.groupby("region")["total_pickups_model"]
    .transform(lambda s: s.rolling(window=rolling_window, min_periods=16).std())
)

historical["rolling_mean"] = historical["rolling_mean"].fillna(historical["avg_pickups_ewm"])
historical["rolling_std"] = historical["rolling_std"].fillna(0)

historical["surge_threshold"] = historical["rolling_mean"] + surge_k * historical["rolling_std"]
historical["surge_flag"] = historical["predicted_demand"] > historical["surge_threshold"]

historical["surge_score"] = (
    (historical["predicted_demand"] - historical["rolling_mean"])
    / (historical["rolling_std"] + 1e-6)
)
historical["surge_intensity"] = historical["predicted_demand"] / (historical["surge_threshold"] + 1e-6)

historical["surge_level"] = "none"
historical.loc[historical["surge_flag"] & (historical["surge_intensity"] <= 1.15), "surge_level"] = "low"
historical.loc[
    historical["surge_flag"]
    & (historical["surge_intensity"] > 1.15)
    & (historical["surge_intensity"] <= 1.35),
    "surge_level",
] = "medium"
historical.loc[historical["surge_flag"] & (historical["surge_intensity"] > 1.35), "surge_level"] = "high"

historical["risk_score"] = historical["rolling_std"] / (historical["rolling_mean"] + 1e-6)
historical["risk_band"] = pd.cut(
    historical["risk_score"],
    bins=[-np.inf, 0.35, 0.75, np.inf],
    labels=["Stable", "Moderate", "Volatile"],
).astype(str)


In [ ]:
# Revenue, efficiency, congestion, and pressure features
if "total_revenue" in historical.columns:
    historical["avg_fare_region_slot"] = historical["total_revenue"] / historical["total_pickups_raw"].clip(lower=1)
    region_fare_baseline = historical.groupby("region")["avg_fare_region_slot"].transform("mean")
    historical["avg_fare_region_slot"] = historical["avg_fare_region_slot"].fillna(region_fare_baseline)
    historical["avg_fare_region_slot"] = historical["avg_fare_region_slot"].fillna(historical["avg_fare_region_slot"].median())
else:
    historical["avg_fare_region_slot"] = np.nan

historical["expected_revenue"] = historical["predicted_demand"] * historical["avg_fare_region_slot"]

if "total_trip_distance" in historical.columns and "total_revenue" in historical.columns:
    historical["fare_per_km"] = historical["total_revenue"] / historical["total_trip_distance"].clip(lower=1e-3)
else:
    historical["fare_per_km"] = np.nan

if "total_revenue" in historical.columns and "total_tip" in historical.columns:
    historical["tip_ratio"] = historical["total_tip"] / historical["total_revenue"].clip(lower=1e-3)
    historical["tip_per_pickup"] = historical["total_tip"] / historical["total_pickups_raw"].clip(lower=1)
else:
    historical["tip_ratio"] = np.nan
    historical["tip_per_pickup"] = np.nan

if "total_revenue" in historical.columns:
    historical["revenue_density_15min"] = historical["total_revenue"]
else:
    historical["revenue_density_15min"] = np.nan

region_avg_proxy = historical.groupby("region")["predicted_demand"].transform("mean")
historical["demand_pressure"] = historical["predicted_demand"] / (region_avg_proxy + 1e-6)

# Traffic / speed proxy from available aggregates
if {"total_trip_distance", "avg_trip_duration_min", "total_pickups_raw"}.issubset(historical.columns):
    historical["avg_trip_distance_per_ride"] = (
        historical["total_trip_distance"] / historical["total_pickups_raw"].clip(lower=1)
    )
    historical["avg_speed_kmh"] = (
        historical["avg_trip_distance_per_ride"]
        / (historical["avg_trip_duration_min"].clip(lower=1e-3) / 60.0)
    )
    historical["congestion_band"] = pd.cut(
        historical["avg_speed_kmh"],
        bins=[-np.inf, 12, 22, np.inf],
        labels=["High Congestion", "Moderate", "Low Congestion"],
    ).astype(str)
else:
    historical["avg_speed_kmh"] = np.nan
    historical["congestion_band"] = "unknown"


In [ ]:
# Driver allocation optimization (nearby region demand gain / distance)
if neighbors_path.exists():
    neighbors = pd.read_csv(neighbors_path)

    neighbors = neighbors.rename(
        columns={
            "region_id": "region",
            "neighbor_region_id": "target_region",
        }
    )

    required_neighbor_cols = {"region", "target_region", "distance_km"}
    if not required_neighbor_cols.issubset(neighbors.columns):
        raise ValueError(f"Neighbor file missing columns: {required_neighbor_cols}")

    base = historical[["pickup_slot", "region", "predicted_demand"]].copy()

    candidate_moves = base.merge(neighbors[["region", "target_region", "distance_km"]], on="region", how="left")

    target_demand = base.rename(
        columns={
            "region": "target_region",
            "predicted_demand": "target_predicted_demand",
        }
    )

    candidate_moves = candidate_moves.merge(
        target_demand,
        on=["pickup_slot", "target_region"],
        how="left",
    )

    candidate_moves["target_predicted_demand"] = candidate_moves["target_predicted_demand"].fillna(0)
    candidate_moves["expected_demand_gain"] = (
        candidate_moves["target_predicted_demand"] - candidate_moves["predicted_demand"]
    ).clip(lower=0)
    candidate_moves["relocation_score"] = (
        candidate_moves["expected_demand_gain"] / (candidate_moves["distance_km"] + 1e-3)
    )

    top_moves = (
        candidate_moves.sort_values(
            ["pickup_slot", "region", "relocation_score", "expected_demand_gain"],
            ascending=[True, True, False, False],
        )
        .groupby(["pickup_slot", "region"], as_index=False)
        .head(3)
        .copy()
    )
    top_moves["recommendation_rank"] = top_moves.groupby(["pickup_slot", "region"]).cumcount() + 1

    best_moves = top_moves[top_moves["recommendation_rank"] == 1].copy()

    # If gain is too small, recommend staying in current region.
    min_gain_threshold = 1.0
    low_gain_mask = best_moves["expected_demand_gain"] < min_gain_threshold
    best_moves.loc[low_gain_mask, "target_region"] = np.nan
    best_moves.loc[low_gain_mask, "distance_km"] = np.nan
    best_moves.loc[low_gain_mask, "expected_demand_gain"] = 0.0
    best_moves.loc[low_gain_mask, "relocation_score"] = 0.0

    best_moves = best_moves.rename(
        columns={
            "target_region": "recommended_next_zone",
            "distance_km": "recommended_distance_km",
        }
    )

    historical = historical.merge(
        best_moves[
            [
                "pickup_slot",
                "region",
                "recommended_next_zone",
                "recommended_distance_km",
                "expected_demand_gain",
                "relocation_score",
            ]
        ],
        on=["pickup_slot", "region"],
        how="left",
    )

    historical["recommended_next_zone"] = historical["recommended_next_zone"].astype("Float64")
    historical["recommended_next_zone"] = historical["recommended_next_zone"].round().astype("Int64")
else:
    top_moves = pd.DataFrame()
    best_moves = pd.DataFrame()
    historical["recommended_next_zone"] = pd.Series([pd.NA] * len(historical), dtype="Int64")
    historical["recommended_distance_km"] = np.nan
    historical["expected_demand_gain"] = 0.0
    historical["relocation_score"] = 0.0


In [ ]:
# Best-time recommendation by region (historical slot profile)
slot_profile = (
    historical.groupby(["region", "pickup_day_of_week", "pickup_hour"], as_index=False)["predicted_demand"]
    .mean()
    .rename(columns={"predicted_demand": "avg_predicted_demand"})
)

best_time_recommendations = (
    slot_profile.sort_values(["region", "avg_predicted_demand"], ascending=[True, False])
    .groupby("region", as_index=False)
    .head(3)
    .copy()
)
best_time_recommendations["rank"] = best_time_recommendations.groupby("region").cumcount() + 1

day_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}

best_time_recommendations["day_name"] = best_time_recommendations["pickup_day_of_week"].map(day_names)
best_time_recommendations["best_time_window"] = (
    best_time_recommendations["pickup_hour"].astype(int).map(lambda h: f"{h:02d}:00-{(h + 1) % 24:02d}:00")
)

# Add top recommendation per region into main historical table
region_best_time = (
    best_time_recommendations[best_time_recommendations["rank"] == 1]
    [["region", "day_name", "best_time_window"]]
    .rename(columns={"day_name": "best_day_name"})
)

historical = historical.merge(region_best_time, on="region", how="left")

best_time_recommendations.head(10)


In [ ]:
# Save outputs
historical_output = data_interim / "historical_features.csv"
final_data_output = data_interim / "final_data.csv"  # compatibility with older flow
relocation_output = data_interim / "driver_relocation_recommendations.csv"
best_time_output = data_interim / "best_time_recommendations.csv"
slot_profile_output = data_interim / "region_slot_profile.csv"
ui_output = data_interim / "ui_ready_timeslot_output.csv"
smoothing_metrics_output = data_interim / "smoothing_tuning_metrics.csv"

historical.to_csv(historical_output, index=False)

final_data = historical.rename(columns={"pickup_slot": "tpep_pickup_datetime"}).copy()
final_data.to_csv(final_data_output, index=False)

if not top_moves.empty:
    top_moves.to_csv(relocation_output, index=False)
else:
    pd.DataFrame(columns=[
        "pickup_slot",
        "region",
        "target_region",
        "distance_km",
        "target_predicted_demand",
        "expected_demand_gain",
        "relocation_score",
        "recommendation_rank",
    ]).to_csv(relocation_output, index=False)

best_time_recommendations.to_csv(best_time_output, index=False)
slot_profile.to_csv(slot_profile_output, index=False)
smoothing_metrics.to_csv(smoothing_metrics_output, index=False)

ui_cols = [
    "pickup_slot",
    "region",
    "predicted_demand",
    "surge_flag",
    "surge_level",
    "risk_score",
    "risk_band",
    "expected_revenue",
    "recommended_next_zone",
    "recommended_distance_km",
    "relocation_score",
    "demand_pressure",
    "best_day_name",
    "best_time_window",
    "smoothing_method",
    "selected_ma_window",
    "selected_ewma_alpha",
]
ui_ready = historical[[col for col in ui_cols if col in historical.columns]].copy()
ui_ready.to_csv(ui_output, index=False)

print("Saved files:")
for path in [
    historical_output,
    final_data_output,
    relocation_output,
    best_time_output,
    slot_profile_output,
    ui_output,
    smoothing_metrics_output,
]:
    print("-", path.relative_to(project_root))


In [ ]:
# Quick validation checks
key_cols = [
    "pickup_slot",
    "region",
    "total_pickups_raw",
    "predicted_demand",
    "avg_pickups",
    "smoothing_method",
    "selected_ma_window",
    "selected_ewma_alpha",
    "surge_flag",
    "surge_level",
    "risk_score",
    "risk_band",
    "expected_revenue",
    "recommended_next_zone",
    "relocation_score",
    "demand_pressure",
    "best_time_window",
]

available_cols = [col for col in key_cols if col in historical.columns]
historical[available_cols].head(10)


In [ ]:
# High-level stats
print("Rows:", f"{len(historical):,}")
print("Regions:", historical["region"].nunique())
print("Surge rows:", int(historical["surge_flag"].sum()))
print("Average risk score:", round(float(historical["risk_score"].mean()), 4))
print("Average expected revenue:", round(float(historical["expected_revenue"].fillna(0).mean()), 2))
print("Rows with relocation recommendation:", int(historical["recommended_next_zone"].notna().sum()))
print("Selected smoothing method:", selected_smoothing_method)
print("Best MA window:", best_ma_window, "| Best MA MAPE:", round(float(best_ma_row["mape"]), 4))
print("Best EWMA alpha:", best_ewma_alpha, "| Best EWMA MAPE:", round(float(best_ewma_row["mape"]), 4))


## Notes
- This notebook creates business-ready historical features before model training.
- `predicted_demand_proxy` is a temporary proxy derived from tuned smoothing history; replace with real model predictions in later notebooks for production scoring.
- Smoothing is tuned via validation metrics (`MAPE`, `MAE`, `RMSE`, `sMAPE`) across moving-average and EWMA candidates.
- `final_data.csv` is preserved for compatibility with the earlier pipeline.
- `ui_ready_timeslot_output.csv` provides per-region, per-slot features for direct dashboard/API integration.